In [45]:
!pip install stim

In [46]:
import stim

In [47]:
c = stim.Circuit("""
H 0
CNOT 0 1
M 0 1
""")

In [48]:
c

stim.Circuit('''
    H 0
    CX 0 1
    M 0 1
''')

In [49]:
c.compile_sampler().sample(1)

array([[False, False]])

In [50]:
c.compile_sampler().sample(20)

array([[False, False],
       [ True,  True],
       [ True,  True],
       [False, False],
       [False, False],
       [False, False],
       [ True,  True],
       [False, False],
       [ True,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True],
       [False, False],
       [ True,  True],
       [False, False],
       [False, False],
       [ True,  True],
       [False, False],
       [False, False]])

In [51]:
def rep_code(distance, rounds, noise):
  circuit = stim.Circuit()
  qubits = range(2*distance + 1)
  data = qubits[::2]
  measure = qubits[1::2]
  pairs1 = qubits[:-1]
  circuit.append_operation("X_ERROR", data, noise)
  circuit.append_operation("CNOT", pairs1)
  pairs2 = qubits[1:][::-1]
  circuit.append_operation("CNOT", pairs2)
  circuit.append_operation("MR", measure)
  return circuit * rounds

In [52]:
rep_code(distance=3, rounds=2, noise=0.01)

stim.Circuit('''
    REPEAT 2 {
        X_ERROR(0.01) 0 2 4 6
        CX 0 1 2 3 4 5 6 5 4 3 2 1
        MR 1 3 5
    }
''')

In [53]:
rep_code(distance=3, rounds=2, noise=0.1).compile_sampler().sample(5)

array([[False,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False],
       [ True,  True, False, False,  True, False],
       [False, False,  True, False, False,  True],
       [False, False, False,  True,  True, False]])

In [54]:
def shot(circuit, distance, rounds):
    sample = circuit.compile_sampler().sample(1)[0]

    width = distance - 1  # detectors per round

    for r in range(rounds):
        chunk = sample[r * width:(r + 1) * width]
        shifted = list(chunk[r:]) + [0]*r  # pad right
        print("".join("_1"[int(e)] for e in shifted))

In [55]:
import shutil

In [56]:
shutil.get_terminal_size().columns

80

In [57]:
nc = shutil.get_terminal_size().columns

In [58]:
shot(rep_code(distance=nc, rounds=20, noise=0.01), distance=nc, rounds=20)

_______________________________________________________________________________
_______________________11___________________________11_________________________
_______________________11___________________________11_________________________
_______________________11___________________________11_________________________
_______11______________11___________________________11_________________________
_______11______________11___________________________11_________________________
___11__11______________11__________________11_______11_________________________
___11__11______________11__________________11_______11_________________________
___11__11______________11__________________11_______11_________________________
___11__11______________11__________________11_______11_________________________
___11__11______________11__________________11_______11_________________________
___11__11______________11__________________11_______1_1________________________
___11__11______________11_______________

In [59]:
def rep_code(distance, rounds, noise):
  circuit = stim.Circuit()
  qubits = range(2*distance + 1)
  data = qubits[::2]
  measure = qubits[1::2]

  pairs1 = qubits[:-1]
  circuit.append_operation("CNOT", pairs1)
  circuit.append_operation("DEPOLARIZE2", pairs1, noise)

  pairs2 = qubits[1:][::-1]
  circuit.append_operation("CNOT", pairs2)
  circuit.append_operation("DEPOLARIZE2", pairs2, noise)

  circuit.append_operation("DEPOLARIZE1", qubits, noise)
  circuit.append_operation("MR", measure)

  return circuit * rounds

In [60]:
rep_code(distance=3, rounds=2, noise=0.01)

stim.Circuit('''
    REPEAT 2 {
        CX 0 1 2 3 4 5
        DEPOLARIZE2(0.01) 0 1 2 3 4 5
        CX 6 5 4 3 2 1
        DEPOLARIZE2(0.01) 6 5 4 3 2 1
        DEPOLARIZE1(0.01) 0 1 2 3 4 5 6
        MR 1 3 5
    }
''')

In [61]:
shot(rep_code(distance=nc, rounds=20, noise=0.01), distance=nc, rounds=20)

________________________________________________________1______________________
_________11________________________1_______________________11__________________
_________11________________________11_________1_1__________11_______________1__
_________11_______________11_______11__________________1___11__________________
_________11_______11______11_______11______________________11__1_______________
1_______1_1_______11______11_______11______________________11__________________
1_______1_1_______11______11_______11______________________11__________________
111_____1_1_______11______11_______11______________________11__________________
11_1____1_1_______11______11________1______________________11__________________
11_1____1_1_______11______11____11_11______________________11__________________
11_1____1_1_______111_____11__1_11_11______________________11__________________
11_1____1_1_______11______11____11_11______________________11__________________
11_1____1_1_______11______1_1___11_11___

In [62]:
def rep_code(distance, rounds, noise):
  circuit = stim.Circuit()
  qubits = range(2*distance + 1)
  data = qubits[::2]
  measure = qubits[1::2]

  pairs1 = qubits[:-1]
  circuit.append_operation("CNOT", pairs1)
  circuit.append_operation("DEPOLARIZE2", pairs1, noise)

  pairs2 = qubits[1:][::-1]
  circuit.append_operation("CNOT", pairs2)
  circuit.append_operation("DEPOLARIZE2", pairs2, noise)

  circuit.append_operation("DEPOLARIZE1", qubits, noise)
  circuit.append_operation("MR", measure)

  circuit.append_operation("DETECTOR", [stim.target_rec(-1), stim.target_rec(-1-distance)])

  return circuit * rounds

In [63]:
shot(rep_code(distance=nc, rounds=20, noise=0.01), distance=nc, rounds=20)

__________________1______________________________________1_____________________
__________1_1____________________________________________11___1____________1___
__________1_1_____11_11_____________________________1____11________________11__
____________1_____11_11___________________1_11___________11________________1___
___________11____1_11_1_________11__11____1111_11________11____________________
___________11____1_11_1_________11__11____1111_11________11____________________
___________11____1_11_1_________11__11____1111_11____11__11__1_________________
___________11____1_11_1_____111_1___11____1111_1_____11__11_11_________________
___________11____1111_1_1____11_1_1_11____1111_1111__11__11_11___11____________
___________11__1_1_11_1_11___11_1_1_11____111__11____11__11_11___11____________
___________11__111_11_1_11___11_1_1_11____1111_11____11__11_1_1__11____________
___________11__111_11_1_11___11_1_1_1_1___1111_11____11___1_1_1__11____________
___________11__111_11_1_11___11_1_1_11__

In [64]:
def detector_shot(circuit, distance, rounds):
    sample = circuit.compile_sampler().sample(1)[0]

    width = distance - 1  # detectors per round

    for r in range(rounds):
        chunk = sample[r * width:(r + 1) * width]
        shifted = list(chunk[r:]) + [0]*r  # pad right
        print("".join("_1"[int(e)] for e in shifted))

In [66]:
detector_shot(rep_code(distance=nc, rounds=20, noise=0.01), distance=nc, rounds=20)

___________________11__________________________________________________________
___________________11_______________________1_______11_________________________
____1______________11_____________11______1__11_____11____________1____________
___________________11_______1_____11______11_11_____11_________________________
______________1____11_____________11______11_11_____11_________________________
____1______________11_____________11______11_11_____11___________11____________
_11________________11_____1_______11______11_11_____11__11_______11____________
_11_1______________11_____11______11______11_11_____11__11_______11____________
_11__________11____11_____11______11______11_11_____11__11_______11____________
_11__________11____11_____11______11______11_11_____11__11______1_1_11_________
_11__________11____11_____11__111_11______11_11_____11__11______11__1__________
_11__________11____11_____11__11__11______11_11_____11__11______11_1___________
_11__________11____11_____11__11__11____